# Level 3 — 불균형 대응 및 고급 Augmentation

**목표**: 다수 클래스의 정확도를 크게 희생하지 않으면서, 소수 클래스 (foggy / snowy / dawn-dusk) 의 성능을 끌어올립니다.

다음 축에서 **최소 2가지 이상** 의 기법을 적용하세요.
- Loss-level: Weighted CE, Focal Loss, LDAM, Class-Balanced Loss
- Sampling-level: class-balanced sampler
- Augmentation-level: RandAugment, Mixup, CutMix

Level 1 / 2 에서 가장 좋았던 백본을 base 로 사용하세요. wandb 를 사용하면 여러 기법의 비교 Run 을 같은 프로젝트에 모아 볼 수 있어 편리합니다.

In [ ]:
import os
import sys

# 1. 코랩 환경에서 레포지토리가 클론되지 않은 경우에만 Clone 진행
repo_name = "2026-HYU-AUE8088-PA2"
if not os.path.exists(f"/content/{repo_name}"):
    !git clone https://github.com/dahye411/2026-HYU-AUE8088-PA2.git

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.datasets.samplers import class_balanced_sampler
from src.losses.imbalanced import FocalLoss, ClassBalancedLoss, LDAMLoss, weighted_cross_entropy
from src.augment.mix import mixup_data, cutmix_data, mixed_loss
from src.models.vit import vit_small_patch16_224

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level3"]

# 아래 index만 바꿔가며 셀을 다시 실행
# 0: CE baseline
# 1: Focal Loss
# 2: CE + weather-balanced sampler
# 3: Focal Loss + weather-balanced sampler
# 4: Focal Loss + weather-balanced sampler + Mixup/CutMix
EXPERIMENTS = [
    {"name": "ce_baseline", "loss": "ce", "use_sampler": False, "use_mix": False},
    {"name": "focal", "loss": "focal", "use_sampler": False, "use_mix": False},
    {"name": "weather_sampler", "loss": "ce", "use_sampler": True, "use_mix": False},
    {"name": "focal_weather_sampler", "loss": "focal", "use_sampler": True, "use_mix": False},
    {"name": "focal_weather_sampler_mix", "loss": "focal", "use_sampler": True, "use_mix": True},
]

CURRENT_EXP_INDEX = 0
CURRENT_EXP = EXPERIMENTS[CURRENT_EXP_INDEX]
EXPERIMENT_NAME = CURRENT_EXP["name"]
print("Current experiment:", CURRENT_EXP)

In [ ]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())


# weather 기준 class-balanced sampler: foggy/snowy 등 weather 소수 클래스를 더 자주 관측
if CURRENT_EXP["use_sampler"]:
    sampler = class_balanced_sampler(train_ds, attribute="weather")
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH,
        sampler=sampler,
        num_workers=2,
        worker_init_fn=seed_worker,
        pin_memory=True,
    )
else:
    sampler = None
    g = torch.Generator(); g.manual_seed(SEED)
    train_loader = DataLoader(
        train_ds,
        batch_size=BATCH,
        shuffle=True,
        num_workers=2,
        worker_init_fn=seed_worker,
        generator=g,
        pin_memory=True,
    )

val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print("sampler:", "class_balanced(weather)" if CURRENT_EXP["use_sampler"] else "shuffle")

In [ ]:
# 옵션 B — CE 또는 Focal Loss 를 적용하고 ImageNet pretrained ViT-S/16 base model 을 준비
samples_w = train_ds.class_counts("weather")
samples_s = train_ds.class_counts("scene")
samples_t = train_ds.class_counts("timeofday")

if CURRENT_EXP["loss"] == "ce":
    loss_fns = {a: nn.CrossEntropyLoss().to(device) for a in ATTRIBUTES}
elif CURRENT_EXP["loss"] == "focal":
    loss_fns = {a: FocalLoss(gamma=2.0).to(device) for a in ATTRIBUTES}
else:
    raise ValueError(f"Unknown loss: {CURRENT_EXP['loss']}")

USE_PRETRAINED = True
PRETRAINED_PATH = "./pretrained_checkpoints/deit_small_patch16_224-cd65a155.pth"
PRETRAINED_SOURCE = "https://huggingface.co/facebook/deit-small-patch16-224"


# 현재 ViT 구현체와 이름 및 shape이 일치하는 pretrained tensor만 반환
def remap_vit_pretrained(pre, model):
    if isinstance(pre, dict) and "state_dict" in pre:
        pre = pre["state_dict"]
    elif isinstance(pre, dict) and "model" in pre:
        pre = pre["model"]

    model_state = model.state_dict()
    mapped = {}

    for k, v in pre.items():
        if not torch.is_tensor(v):
            continue

        k = k.removeprefix("module.")
        k = k.removeprefix("backbone.")
        k = k.removeprefix("model.")

        k = k.replace(".mlp.fc1.", ".mlp.0.")
        k = k.replace(".mlp.fc2.", ".mlp.3.")

        if k.startswith("head."):
            continue

        if k in model_state and model_state[k].shape == v.shape:
            mapped[k] = v

    return mapped


model = vit_small_patch16_224()
matched_pretrained_tensors = 0

if USE_PRETRAINED:
    pre = torch.load(PRETRAINED_PATH, map_location="cpu")
    mapped = remap_vit_pretrained(pre, model)
    matched_pretrained_tensors = len(mapped)
    missing, unexpected = model.load_state_dict(mapped, strict=False)
else:
    print("ViT-S/16을 scratch부터 학습")

model = model.to(device)
epochs = 30
LR = 5e-4
WEIGHT_DECAY = 5e-2
optim  = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level3-vit_s16_pretrained-{EXPERIMENT_NAME}",
    config={
        "backbone": "vit_s16",
        "pretrained": USE_PRETRAINED,
        "pretrained_source": PRETRAINED_SOURCE if USE_PRETRAINED else None,
        "matched_pretrained_tensors": matched_pretrained_tensors,
        "sampler": "class_balanced(weather)" if CURRENT_EXP["use_sampler"] else "shuffle",
        "loss": CURRENT_EXP["loss"],
        "mix": CURRENT_EXP["use_mix"],
        "epochs": epochs, "batch": BATCH, "lr": LR, "weight_decay": WEIGHT_DECAY, "seed": SEED,
    },
    tags=WANDB_TAGS + ["vit_s16", "pretrained", EXPERIMENT_NAME],
)
trainer = MultiTaskTrainer(model, optim, sched, loss_fns, device, TrainConfig(epochs=epochs), logger=logger)
print("backbone: vit_s16 pretrained")
print("loss:", CURRENT_EXP["loss"])


In [ ]:
# 옵션 C — 일반 trainer.fit 또는 Mixup/CutMix 학습 루프를 실행
from tqdm import tqdm


def step_with_mix(images, targets):
    """50% 확률로 Mixup, 나머지 50% 확률로 CutMix 적용."""
    if torch.rand(1).item() < 0.5:
        x, ya, yb, lam = mixup_data(images, targets, alpha=0.2)
    else:
        x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)
    logits = model(x)
    return mixed_loss(loss_fns, logits, ya, yb, lam)


if not CURRENT_EXP["use_mix"]:
    history = trainer.fit(train_loader, val_loader)
else:
    history = {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []}
    scaler = torch.amp.GradScaler(enabled=(device.type == "cuda"))

    for epoch in range(epochs):
        model.train()
        running, n_batches = 0.0, 0
        pbar = tqdm(train_loader, desc=f"mix e{epoch+1}", leave=False)
        for batch in pbar:
            images = batch["image"].to(device, non_blocking=True)
            targets = {a: batch[a].to(device, non_blocking=True) for a in ATTRIBUTES}

            optim.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda", enabled=(device.type == "cuda")):
                loss = step_with_mix(images, targets)

            scaler.scale(loss).backward()
            scaler.unscale_(optim)
            nn.utils.clip_grad_norm_(model.parameters(), trainer.cfg.grad_clip)
            scaler.step(optim)
            scaler.update()

            running += loss.item()
            n_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        train_loss = running / max(n_batches, 1)
        val_metrics = trainer.evaluate(val_loader)
        sched.step()

        history["train_loss"].append(train_loss)
        history["val_avg_mf1"].append(val_metrics["avg_macro_f1"])
        history["val_per_mf1"].append(val_metrics["per_macro_f1"])

        log_payload = {
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "val/avg_macro_f1": val_metrics["avg_macro_f1"],
            "lr": optim.param_groups[0]["lr"],
        }
        for a, v in val_metrics["per_macro_f1"].items():
            log_payload[f"val/mf1_{a}"] = v
        logger.log(log_payload, step=epoch)

        print(
            f"[epoch {epoch+1:02d}/{epochs}] "
            f"train_loss={train_loss:.4f}  "
            f"val_avg_MF1={val_metrics['avg_macro_f1']:.4f}  "
            f"per={val_metrics['per_macro_f1']}"
        )

In [ ]:
# 학습 종료 후 — 속성별 confusion matrix + per-class F1 표를 wandb 에 업로드
val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)
prf = per_class_prf(val_pred, val_tgt)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    rows = list(zip(prf[a]["class"], prf[a]["precision"], prf[a]["recall"], prf[a]["f1"], prf[a]["support"]))
    logger.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"], [list(r) for r in rows])
logger.finish()

os.makedirs("../checkpoints", exist_ok=True)
torch.save(
    {
        "state_dict": model.state_dict(),
        "history": history,
        "prf": prf,
        "config": CURRENT_EXP,
        "backbone": "vit_s16",
        "pretrained": USE_PRETRAINED,
        "pretrained_source": PRETRAINED_SOURCE if USE_PRETRAINED else None,
        "matched_pretrained_tensors": matched_pretrained_tensors,
    },
    f"../checkpoints/level3_{EXPERIMENT_NAME}.pth",
)
print("saved:", f"../checkpoints/level3_{EXPERIMENT_NAME}.pth")

## 분석 (필수)

각 기법에 대해 **속성별 per-class F1 표** 를 작성하세요. 다음 항목을 강조해 주세요.
- 소수 클래스 (foggy / snowy / dawn-dusk) 의 적용 전후 성능 차이.
- 다수 클래스의 회귀 (regression) 발생 여부 — 그 trade-off 가 정당한지 논거.
- Sampling 과 Mixup / CutMix 의 상호작용 — 서로 도움이 되는지 충돌하는지.